In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import re
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

In [2]:
df = pd.read_csv('/content/Dataset---Hate-Speech-Detection-using-Deep-Learning.csv')
df.head()

,class,tweet
0,2,!!! RT @mayasolovely: As a woman you shouldn't...
1,1,!!!!! RT @mleew17: boy dats cold...tyga dwn ba...
2,1,!!!!!!! RT @UrKindOfBrand Dawg!!!! RT @80sbaby...
3,1,!!!!!!!!! RT @C_G_Anderson: @viva_based she lo...
4,1,!!!!!!!!!!!!! RT @ShenikaRoberts: The shit you...


In [3]:
df['class'].value_counts()

,count
class,
1,19190
2,4163
0,1430


In [4]:
print(df.isnull().sum())

class    0
tweet    0
dtype: int64


In [5]:
print(df.duplicated().sum())

0


In [6]:
df['text_length']= df['tweet'].apply(lambda x: len(x.split()))

print('Max Length:' , df['text_length'].max())

print('Min Length:' , df['text_length'].min())

print('Average Length:' , df['text_length'].mean())

df.sort_values(
    by='text_length',
    ascending = False
)[['tweet','text_length']].head(10)

Max Length: 52
Min Length: 1
Average Length: 14.117015696243392


,tweet,text_length
22478,Was finna slit my eyebrows up in the shop but ...,52
2099,"' She herd I was a dope boy , I herd she was a...",36
10377,I got no pic up lines I stay on my grind I tel...,33
11383,"I'm sippin on Patron nigga , my bitch bad to t...",33
19115,RT @hspiotta_21: c is for cunt\nu is for ur a ...,33
13699,"Oomf a hoe. She know she a hoe, I know she a h...",33
23174,Yo bitch a freak fucked ha to sleep and dat wa...,33
11158,I'd shut the fuck up or I'll just be wide and ...,33
20418,RT @yung_gleesh: If rather fukk a bitch dat fu...,32
24420,shout out to pullz cuz I kno if I was trash he...,32


In [7]:
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('omw-1.4')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def clean_text(text):
  text= text.lower()

  text = re.sub(r"http\S+\www\S+" , "", text)

  text = re.sub(r"<.*?>", "", text)

  text = re.sub(r"[^a-z\s]", "", text)

  words = text.split()
  words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]

  text = " ".join(words)

  text = re.sub(r"\s+", " ", text)

  return text.strip()

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [8]:
df['tweet'] = df['tweet'].apply(clean_text)

In [9]:
sentences = df['tweet'].values
labels = df['class'].values

X_train, X_test, y_train, y_test = train_test_split(
    sentences,
    labels,
    test_size = 0.2,
    random_state = 42,
    stratify = labels
)

In [10]:
vocab_size = 7000
oov_token = "<OOV>"

tokenizer = Tokenizer(
    num_words = vocab_size,
    oov_token = oov_token
)

tokenizer.fit_on_texts(X_train)

In [11]:
train_sequences = tokenizer.texts_to_sequences(X_train)

test_sequences = tokenizer.texts_to_sequences(X_test)

In [15]:
max_length = 40

X_train_pad = pad_sequences(
    train_sequences,
    maxlen = max_length,
    padding='post',
    truncating = 'post'
)

X_test_pad = pad_sequences(
    test_sequences,
    maxlen = max_length,
    padding='post',
    truncating = 'post'
)

In [16]:
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler

ros = RandomOverSampler(sampling_strategy={0: 5000, 2: 7000},random_state = 42)
X_train_pad, y_train = ros.fit_resample(X_train_pad, y_train)

rus = RandomUnderSampler(sampling_strategy={1: 11000}, random_state=42)
X_train_pad, y_train = rus.fit_resample(X_train_pad, y_train)

print(pd.Series(y_train).value_counts())

1    11000
2     7000
0     5000
Name: count, dtype: int64


In [17]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
class_weights_dict = dict(zip(classes, weights))

In [18]:
from tensorflow.keras.layers import SimpleRNN
embedding_dim = 128

modelRNN = Sequential([

                     Input(shape=(max_length,)),

                     Embedding(
                         input_dim = vocab_size,
                         output_dim = embedding_dim
                     ),

                     SimpleRNN(64),

                     Dropout(0.3),

                     Dense(32, activation='relu'),

                     Dropout(0.3),

                     Dense(3, activation='softmax')
])

In [19]:
modelRNN.compile(
    optimizer = Adam(learning_rate=0.0005),
    loss = 'sparse_categorical_crossentropy',
    metrics = ['accuracy']
)

modelRNN.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (None, 40, 128)        │       896,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ (None, 64)             │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 910,531 (3.47 MB)

 Trainable params: 910,531 (3.47 MB)

 Non-trainable params: 0 (0.00 B)

In [20]:
historyRNN = modelRNN.fit(
    X_train_pad,
    y_train,
    epochs = 10,
    batch_size = 64,
    validation_data = (X_test_pad, y_test),
    class_weight = class_weights_dict,
    callbacks = [
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3)
    ]
)

Epoch 1/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 17s 26ms/step - accuracy: 0.6940 - loss: 0.7104 - val_accuracy: 0.7688 - val_loss: 0.6083 - learning_rate: 5.0000e-04
Epoch 2/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.8749 - loss: 0.3306 - val_accuracy: 0.8031 - val_loss: 0.5535 - learning_rate: 5.0000e-04
Epoch 3/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9366 - loss: 0.1859 - val_accuracy: 0.8092 - val_loss: 0.6184 - learning_rate: 5.0000e-04
Epoch 4/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9579 - loss: 0.1254 - val_accuracy: 0.8114 - val_loss: 0.6860 - learning_rate: 5.0000e-04
Epoch 5/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9649 - loss: 0.1057 - val_accuracy: 0.8112 - val_loss: 0.7326 - learning_rate: 5.0000e-04
Epoch 6/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9807 - loss: 0.0635 - val_accuracy: 0.8356 - val_loss: 0.7539 - learning_rate: 2.5000e-04
Epoch 7/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy

In [21]:
y_pred = modelRNN.predict(X_test_pad)
y_pred_classes = np.argmax(y_pred, axis=1)

print(classification_report(y_test, y_pred_classes))
print(confusion_matrix(y_test, y_pred_classes))

155/155 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
              precision    recall  f1-score   support

           0       0.22      0.36      0.27       286
           1       0.94      0.83      0.88      3838
           2       0.64      0.85      0.73       833

    accuracy                           0.80      4957
   macro avg       0.60      0.68      0.63      4957
weighted avg       0.85      0.80      0.82      4957

[[ 104  110   72]
 [ 350 3168  320]
 [  26   98  709]]


In [22]:
model = Sequential([

                     Input(shape=(max_length,)),

                     Embedding(
                         input_dim = vocab_size,
                         output_dim = embedding_dim
                     ),

                     LSTM(64),

                     Dropout(0.3),

                     Dense(32, activation='relu'),

                     Dropout(0.3),

                     Dense(3, activation='softmax')
])

In [23]:
model.compile(
    optimizer = Adam(learning_rate = 0.0005),
    loss = 'sparse_categorical_crossentropy',
    metrics =["accuracy"]
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 40, 128)        │       896,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 947,587 (3.61 MB)

 Trainable params: 947,587 (3.61 MB)

 Non-trainable params: 0 (0.00 B)

In [24]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

historyLSTM =model.fit(
    X_train_pad,
    y_train,
    epochs = 10,
    batch_size = 64,
    validation_data = (X_test_pad, y_test),
    callbacks =[
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3)
    ]
)

Epoch 1/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - accuracy: 0.6486 - loss: 0.7790 - val_accuracy: 0.8152 - val_loss: 0.4803 - learning_rate: 5.0000e-04
Epoch 2/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.8497 - loss: 0.4372 - val_accuracy: 0.8461 - val_loss: 0.3972 - learning_rate: 5.0000e-04
Epoch 3/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9092 - loss: 0.2918 - val_accuracy: 0.8735 - val_loss: 0.3938 - learning_rate: 5.0000e-04
Epoch 4/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9374 - loss: 0.2171 - val_accuracy: 0.8667 - val_loss: 0.4298 - learning_rate: 5.0000e-04
Epoch 5/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9523 - loss: 0.1721 - val_accuracy: 0.8410 - val_loss: 0.5423 - learning_rate: 5.0000e-04
Epoch 6/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9593 - loss: 0.1516 - val_accuracy: 0.8580 - val_loss: 0.5112 - learning_rate: 5.0000e-04
Epoch 7/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy

In [25]:
loss, accuracy = model.evaluate(X_test_pad, y_test)

print('Loss:', loss)
print('Accuracy:', accuracy)

155/155 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8735 - loss: 0.3938
Loss: 0.3937922716140747
Accuracy: 0.8735122084617615


In [26]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test_pad)

y_pred_classes = np.argmax(y_pred, axis=1)

print(classification_report(y_test, y_pred_classes))
print(confusion_matrix(y_test, y_pred_classes))

155/155 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
              precision    recall  f1-score   support

           0       0.38      0.41      0.39       286
           1       0.94      0.90      0.92      3838
           2       0.77      0.90      0.83       833

    accuracy                           0.87      4957
   macro avg       0.70      0.74      0.71      4957
weighted avg       0.88      0.87      0.88      4957

[[ 117  134   35]
 [ 187 3466  185]
 [   6   80  747]]


In [27]:
from tensorflow.keras.layers import Bidirectional

model2 = Sequential([

                     Input(shape=(max_length,)),

                     Embedding(
                         input_dim = vocab_size,
                         output_dim = embedding_dim
                     ),

                     Bidirectional(
                     LSTM(64)
                     ),

                     Dropout(0.3),

                     Dense(32, activation='relu'),

                     Dropout(0.3),

                     Dense(3, activation='softmax')
])

In [28]:
model2.compile(
    optimizer = Adam(learning_rate=0.0005),
    loss ='sparse_categorical_crossentropy',
    metrics =['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 40, 128)        │       896,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,842,763 (10.84 MB)

 Trainable params: 947,587 (3.61 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,895,176 (7.23 MB)

In [30]:
historyBi =model2.fit(
    X_train_pad,
    y_train,
    epochs = 10,
    batch_size = 64,
    validation_data = (X_test_pad, y_test),
    class_weight = {0: 2.0, 1: 1.0, 2: 1.2},
    callbacks =[
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3)
    ]
)

Epoch 1/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.7333 - loss: 0.8511 - val_accuracy: 0.8521 - val_loss: 0.3940 - learning_rate: 5.0000e-04
Epoch 2/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9070 - loss: 0.3506 - val_accuracy: 0.8527 - val_loss: 0.4098 - learning_rate: 5.0000e-04
Epoch 3/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9415 - loss: 0.2258 - val_accuracy: 0.8326 - val_loss: 0.5228 - learning_rate: 5.0000e-04
Epoch 4/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9578 - loss: 0.1661 - val_accuracy: 0.8533 - val_loss: 0.5260 - learning_rate: 5.0000e-04
Epoch 5/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9746 - loss: 0.1133 - val_accuracy: 0.8497 - val_loss: 0.6482 - learning_rate: 2.5000e-04
Epoch 6/10
360/360 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.9786 - loss: 0.0919 - val_accuracy: 0.8652 - val_loss: 0.6770 - learning_rate: 2.5000e-04


In [31]:
loss, accuracy = model2.evaluate(X_test_pad, y_test)

print('Loss:', loss)
print('Accuracy:', accuracy)

155/155 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8521 - loss: 0.3940
Loss: 0.3939701318740845
Accuracy: 0.8521283268928528


In [32]:
y_pred = model2.predict(X_test_pad)

y_pred_classes = np.argmax(y_pred, axis=1)

print(classification_report(y_test, y_pred_classes))
print(confusion_matrix(y_test, y_pred_classes))

155/155 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
              precision    recall  f1-score   support

           0       0.31      0.62      0.41       286
           1       0.96      0.86      0.91      3838
           2       0.80      0.91      0.85       833

    accuracy                           0.85      4957
   macro avg       0.69      0.79      0.72      4957
weighted avg       0.89      0.85      0.87      4957

[[ 176   93   17]
 [ 373 3292  173]
 [  25   52  756]]


In [33]:
model2.save('bilstm_hate_speech.keras')

import pickle

with open('tokenizer.pkl', 'wb') as f:
    pickle.dump(tokenizer, f)

In [34]:
from google.colab import files

files.download('bilstm_hate_speech.keras')
files.download('tokenizer.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>